In [ ]:
!pip install -q yfinance pandas-datareader

In [ ]:
TICKERS = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA"]
START_DATE = "2026-06-01"
END_DATE = "2026-08-01"

# A price difference larger than this (as a fraction of the Yahoo price)
# is flagged as a mismatch rather than treated as normal rounding noise.
TOLERANCE_PCT = 0.005  # 0.5%

print(f"Comparing {len(TICKERS)} tickers from {START_DATE} to {END_DATE}")

Comparing 5 tickers from 2026-06-01 to 2026-08-01


In [ ]:
import yfinance as yf
import pandas as pd

def fetch_yahoo(tickers, start, end):
    rows = []
    for t in tickers:
        data = yf.download(t, start=start, end=end, progress=False, auto_adjust=False)
        if data.empty:
            print(f"no data for {t}")
            continue
        if hasattr(data.columns, "nlevels") and data.columns.nlevels > 1:
            data.columns = data.columns.get_level_values(0)
        for idx, row in data.iterrows():
            rows.append({
                "ticker": t,
                "price_date": idx.strftime("%Y-%m-%d"),
                "close_price": round(float(row["Close"]), 4),
            })
    return pd.DataFrame(rows)

df_yahoo = fetch_yahoo(TICKERS, START_DATE, END_DATE)
print(len(df_yahoo), "rows")
df_yahoo.head()

215 rows


,ticker,price_date,close_price
0,AAPL,2026-06-01,306.31
1,AAPL,2026-06-02,315.20
2,AAPL,2026-06-03,310.26
3,AAPL,2026-06-04,311.23
4,AAPL,2026-06-05,307.34


In [ ]:
ALPHA_VANTAGE_KEY = "1NJ3XEGEK1I1I47X"

In [ ]:
def fetch_alpha_vantage(tickers, start, end, api_key):
    rows = []
    for t in tickers:
        url = "https://www.alphavantage.co/query"
        params = {
            "function": "TIME_SERIES_DAILY",
            "symbol": t,
            "outputsize": "compact",
            "apikey": api_key,
        }
        resp = requests.get(url, params=params, timeout=15)
        data = resp.json()

        series = data.get("Time Series (Daily)")
        if not series:
            print(f"no alpha vantage data for {t}: {data}")
            continue

        for date_str, values in series.items():
            if start <= date_str <= end:
                rows.append({
                    "ticker": t,
                    "price_date": date_str,
                    "close_price": round(float(values["4. close"]), 4),
                })
        time.sleep(13)

    return pd.DataFrame(rows)

df_stooq = fetch_alpha_vantage(TICKERS, START_DATE, END_DATE, ALPHA_VANTAGE_KEY)
print(len(df_stooq), "rows")
df_stooq.head()

215 rows


,ticker,price_date,close_price
0,AAPL,2026-07-31,308.91
1,AAPL,2026-07-30,333.43
2,AAPL,2026-07-29,338.19
3,AAPL,2026-07-28,340.08
4,AAPL,2026-07-27,336.91


In [ ]:
df_alpha = df_stooq.copy()  # renaming since this is Alpha Vantage data, not Stooq

import sqlite3

conn = sqlite3.connect(":memory:")
df_yahoo.to_sql("source_yahoo", conn, index=False, if_exists="replace")
df_alpha.to_sql("source_alpha", conn, index=False, if_exists="replace")

query = f"""
SELECT
    a.ticker,
    a.price_date,
    a.close_price AS yahoo_close,
    b.close_price AS alpha_close,
    ROUND(b.close_price - a.close_price, 4) AS price_diff,
    ROUND(100.0 * (b.close_price - a.close_price) / a.close_price, 4) AS pct_diff,
    CASE
        WHEN ABS(b.close_price - a.close_price) / a.close_price > {TOLERANCE_PCT}
        THEN 'MISMATCH' ELSE 'MATCH'
    END AS status
FROM source_yahoo a
JOIN source_alpha b
    ON a.ticker = b.ticker AND a.price_date = b.price_date
ORDER BY a.ticker, a.price_date;
"""

detail = pd.read_sql(query, conn)
print(len(detail), "rows compared")
detail.head(10)

215 rows compared


,ticker,price_date,yahoo_close,alpha_close,price_diff,pct_diff,status
0,AAPL,2026-06-01,306.31,306.31,0.0,0.0,MATCH
1,AAPL,2026-06-02,315.20,315.20,0.0,0.0,MATCH
2,AAPL,2026-06-03,310.26,310.26,0.0,0.0,MATCH
3,AAPL,2026-06-04,311.23,311.23,0.0,0.0,MATCH
4,AAPL,2026-06-05,307.34,307.34,0.0,0.0,MATCH
5,AAPL,2026-06-08,301.54,301.54,0.0,0.0,MATCH
6,AAPL,2026-06-09,290.55,290.55,0.0,0.0,MATCH
7,AAPL,2026-06-10,291.58,291.58,0.0,0.0,MATCH
8,AAPL,2026-06-11,295.63,295.63,0.0,0.0,MATCH
9,AAPL,2026-06-12,291.13,291.13,0.0,0.0,MATCH


In [ ]:
print(detail["status"].value_counts())
print()
detail[detail["status"] == "MISMATCH"]

status
MATCH    215
Name: count, dtype: int64



,ticker,price_date,yahoo_close,alpha_close,price_diff,pct_diff,status


In [ ]:
gap_query = """
SELECT a.ticker, a.price_date, 'MISSING_IN_ALPHA' AS issue
FROM source_yahoo a
LEFT JOIN source_alpha b ON a.ticker = b.ticker AND a.price_date = b.price_date
WHERE b.ticker IS NULL

UNION ALL

SELECT b.ticker, b.price_date, 'MISSING_IN_YAHOO' AS issue
FROM source_alpha b
LEFT JOIN source_yahoo a ON a.ticker = b.ticker AND a.price_date = b.price_date
WHERE a.ticker IS NULL

ORDER BY 1, 2;
"""

coverage_gaps = pd.read_sql(gap_query, conn)
print(len(coverage_gaps), "coverage gaps")
coverage_gaps

0 coverage gaps


,ticker,price_date,issue


In [ ]:
NEW_TICKERS = ["TSLA", "GME", "CVNA"]

df_yahoo_new = fetch_yahoo(NEW_TICKERS, START_DATE, END_DATE)
print(len(df_yahoo_new), "new yahoo rows")
df_yahoo_new.head()

129 new yahoo rows


,ticker,price_date,close_price
0,TSLA,2026-06-01,415.88
1,TSLA,2026-06-02,423.74
2,TSLA,2026-06-03,423.70
3,TSLA,2026-06-04,418.45
4,TSLA,2026-06-05,391.00


In [ ]:
df_alpha_new = fetch_alpha_vantage(NEW_TICKERS, START_DATE, END_DATE, ALPHA_VANTAGE_KEY)
print(len(df_alpha_new), "new alpha rows")
df_alpha_new.head()

129 new alpha rows


,ticker,price_date,close_price
0,TSLA,2026-07-31,311.21
1,TSLA,2026-07-30,308.85
2,TSLA,2026-07-29,298.32
3,TSLA,2026-07-28,307.44
4,TSLA,2026-07-27,309.22


In [ ]:
df_yahoo = pd.concat([df_yahoo, df_yahoo_new], ignore_index=True)
df_alpha = pd.concat([df_alpha, df_alpha_new], ignore_index=True)

df_yahoo.to_sql("source_yahoo", conn, index=False, if_exists="replace")
df_alpha.to_sql("source_alpha", conn, index=False, if_exists="replace")

detail = pd.read_sql(query, conn)
print(len(detail), "rows compared total")
print(detail["status"].value_counts())

344 rows compared total
status
MATCH    344
Name: count, dtype: int64


In [ ]:
test_yahoo = pd.DataFrame([{"ticker": "TEST", "price_date": "2026-01-01", "close_price": 100.00}])
test_alpha = pd.DataFrame([{"ticker": "TEST", "price_date": "2026-01-01", "close_price": 102.00}])  # 2% off, should trip the 0.5% tolerance

test_yahoo.to_sql("test_yahoo", conn, index=False, if_exists="replace")
test_alpha.to_sql("test_alpha", conn, index=False, if_exists="replace")

test_query = f"""
SELECT a.ticker, a.price_date, a.close_price AS yahoo_close, b.close_price AS alpha_close,
    ROUND(b.close_price - a.close_price, 4) AS price_diff,
    ROUND(100.0 * (b.close_price - a.close_price) / a.close_price, 4) AS pct_diff,
    CASE WHEN ABS(b.close_price - a.close_price) / a.close_price > {TOLERANCE_PCT}
        THEN 'MISMATCH' ELSE 'MATCH' END AS status
FROM test_yahoo a JOIN test_alpha b ON a.ticker = b.ticker AND a.price_date = b.price_date;
"""

pd.read_sql(test_query, conn)

,ticker,price_date,yahoo_close,alpha_close,price_diff,pct_diff,status
0,TEST,2026-01-01,100.0,102.0,2.0,2.0,MISMATCH


In [ ]:
summary_query = f"""
SELECT
    a.ticker,
    COUNT(*) AS days_compared,
    SUM(CASE WHEN ABS(b.close_price - a.close_price) / a.close_price > {TOLERANCE_PCT} THEN 1 ELSE 0 END) AS mismatch_count,
    ROUND(100.0 * SUM(CASE WHEN ABS(b.close_price - a.close_price) / a.close_price > {TOLERANCE_PCT} THEN 1 ELSE 0 END) / COUNT(*), 2) AS mismatch_rate_pct,
    ROUND(AVG(ABS(b.close_price - a.close_price)), 4) AS avg_abs_diff,
    ROUND(MAX(ABS(b.close_price - a.close_price)), 4) AS max_abs_diff
FROM source_yahoo a
JOIN source_alpha b ON a.ticker = b.ticker AND a.price_date = b.price_date
GROUP BY a.ticker
ORDER BY mismatch_rate_pct DESC;
"""

summary = pd.read_sql(summary_query, conn)
summary

,ticker,days_compared,mismatch_count,mismatch_rate_pct,avg_abs_diff,max_abs_diff
0,TSLA,43,0,0.0,0.0,0.0
1,NVDA,43,0,0.0,0.0,0.0
2,MSFT,43,0,0.0,0.0,0.0
3,GOOGL,43,0,0.0,0.0,0.0
4,GME,43,0,0.0,0.0,0.0
5,CVNA,43,0,0.0,0.0,0.0
6,AMZN,43,0,0.0,0.0,0.0
7,AAPL,43,0,0.0,0.0,0.0


In [ ]:
detail.to_csv("reconciliation_detail.csv", index=False)
summary.to_csv("reconciliation_summary.csv", index=False)

from google.colab import files
files.download("reconciliation_detail.csv")
files.download("reconciliation_summary.csv")

print("exported and downloading both csvs")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

exported and downloading both csvs
